# 18 — Seq2seq and the fixed-size bottleneck

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/018-seq2seq.ipynb)

A sequence-to-sequence model reads the whole input into one vector, then writes the output from it. That single vector is the bottleneck, and everything the model can know must fit inside. In this notebook you build a three-number encoder and decoder, plot how thin the bottleneck gets as the sentence grows, and find the length where it fails.

## 1. The numbers: three tokens into three floats

The encoder is the mean of the token vectors, which is the simplest possible read-out. The decoder turns one vector back into the nearest known token by dot product. Three tokens go in and exactly three numbers come out.

In [ ]:
import numpy as np                              # the array library, preinstalled

vectors = {"the": np.array([1.0, 0.0, 0.0]),    # token 0: points along dimension 1
           "cat": np.array([0.0, 1.0, 0.0]),    # token 1: along dimension 2
           "sat": np.array([0.0, 0.0, 1.0])}    # token 2: along dimension 3

def decode(z):                                  # turn one vector back into a token
    return max(vectors, key=lambda w: vectors[w] @ z)   # the token pointing most the same way

sent = ["the", "cat", "sat"]                    # a three-token sentence
z = np.mean([vectors[w] for w in sent], axis=0)  # the encoder: average the token vectors
print("tokens in:", sent)                       # three words
print("bottleneck z:", np.round(z, 3))          # [0.333 0.333 0.333]: three numbers
print("size of z:", z.shape[0], "numbers, whatever the input length")   # constant
print("decoder says:", decode(z))               # 'the' wins a three-way tie

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>vectors = {...}</code> — a one-hot vector per token, easy to reason about.</li><li><code>def decode(z)</code> — the decoder: pick the token whose vector aligns with <code>z</code>.</li><li><code>vectors[w] @ z</code> — dot product, the alignment score.</li><li><code>np.mean([...], axis=0)</code> — average the vectors, position by position.</li><li><code>np.round(z, 3)</code> — round for printing only.</li><li><code>z.shape[0]</code> — how many numbers the bottleneck holds.</li></ul>

</details>

## 2. Step by step: the bottleneck after each token

Feed the words in one at a time and look at the vector after each. It drifts, it never grows, and the decoder's single guess keeps changing because a mean cannot hold three different directions. This is the whole problem in one loop.

In [ ]:
total = np.zeros(3)                             # running sum of token vectors
for i, w in enumerate(sent, start=1):           # i counts tokens seen so far
    total = total + vectors[w]                  # add this token's vector
    z = total / i                               # divide: the running mean
    print(f"after {w:>3}: z = {np.round(z, 2)}  decoder -> {decode(z)}")   # state and guess
print("the three tokens are three different directions:")   # set up the next line
print("no single point can be all three at once")          # the conclusion

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>total = np.zeros(3)</code> — an empty accumulator, same size as one token vector.</li><li><code>enumerate(sent, start=1)</code> — the word and how many tokens have arrived.</li><li><code>total = total + vectors[w]</code> — accumulate, not average yet.</li><li><code>z = total / i</code> — the mean after <code>i</code> tokens.</li><li><code>decode(z)</code> — one guess per prefix, printed next to the state.</li></ul>

</details>

## 3. Plot: bits per token against input length

The bottleneck is three 32-bit floats, so it has 96 bits in total. A longer input does not get a bigger box; it just gets a thinner slice of the same box. The curve falls like `1 / length`, which is the cost of squeezing a sequence into a vector.

In [ ]:
import matplotlib.pyplot as plt                 # plotting library, preinstalled

lengths = np.arange(1, 21)                      # input lengths from 1 to 20 tokens
bits = 3 * 32                                   # three float32 numbers in the bottleneck
per_token = bits / lengths                      # the share each input token can claim
plt.figure(figsize=(6, 3))                      # wide, short figure
plt.plot(lengths, per_token, marker="o", markersize=3, color="tab:blue")   # the shrinking share
plt.yscale("log")                               # log scale: the drop spans two decades
plt.xlabel("input length (tokens)")             # x-axis label
plt.ylabel("bits per input token (log scale)")  # y-axis label
plt.title("96 bits shared by everything: 96 at length 1, 5 at length 20")   # the takeaway
plt.grid(alpha=0.3)                             # faint grid for reading values
plt.show()                                      # render the figure

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>lengths = np.arange(1, 21)</code> — sentence lengths we want to examine.</li><li><code>bits = 3 * 32</code> — three numbers, 32 bits each.</li><li><code>per_token = bits / lengths</code> — the fixed budget divided by length.</li><li><code>plt.yscale("log")</code> — log y-axis, so small values stay visible.</li><li><code>plt.xlabel</code> / <code>plt.ylabel</code> / <code>plt.title</code> — axes and the takeaway.</li></ul>

</details>

## 4. Interactive: lengthen the input

The slider sets how many tokens the encoder reads. The bars are the bottleneck vector and the title shows how many of those tokens the single decoder guess recovers. At length 3 it is already a tie, and it never improves.

In [ ]:
from ipywidgets import interact                   # sliders that rerun a function

def show(n=3):                                  # n = number of tokens
    s = (["cat", "sat", "the"] * n)[:n]         # a sentence of exactly n tokens
    z = np.mean([vectors[w] for w in s], axis=0)  # the encoder: one vector, always
    guess = decode(z)                           # the decoder: one token, always
    acc = s.count(guess) / len(s)               # fraction of the input it names correctly
    plt.figure(figsize=(5.5, 3))                # fresh figure per drag
    plt.bar(["dim 1", "dim 2", "dim 3"], z, color="tab:blue")   # the bottleneck, drawn
    plt.ylim(0, 1)                              # components are shares, so under 1
    plt.xlabel("bottleneck dimension")          # x-axis label
    plt.ylabel("value")                         # y-axis label
    plt.title(f"n = {n}: decoder says '{guess}', correct on {acc * 100:.0f} percent")   # readout
    plt.show()                                  # draw the frame

interact(show, n=(1, 20, 1))                    # slider: 1 to 20 tokens

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>interact</code> — the slider that calls <code>show</code> again.</li><li><code>def show(n=3)</code> — one frame per sequence length.</li><li><code>([...] * n)[:n]</code> — repeat the three tokens, then cut to length n.</li><li><code>np.mean(..., axis=0)</code> — the encoder, unchanged whatever n is.</li><li><code>s.count(guess) / len(s)</code> — how much of the input the one guess covers.</li><li><code>plt.bar([...], z)</code> — the three bottleneck numbers as bars.</li><li><code>interact(show, n=(1, 20, 1))</code> — range and step of the slider.</li></ul>

</details>

## 5. When it breaks: under one float per token

Three floats divided among more than three tokens is less than one number per token, so information must be destroyed. Here twenty tokens share three floats, every token decodes to the same word, and the sentence is unrecoverable. This is the failure an attention mechanism was designed to remove.

In [ ]:
long_sent = (["the", "cat", "sat"] * 7)[:20]    # twenty tokens
z = np.mean([vectors[w] for w in long_sent], axis=0)   # three floats for all of them
print("tokens in:", len(long_sent))             # 20
print("floats in the bottleneck:", z.shape[0])  # 3
print("floats per token:", round(z.shape[0] / len(long_sent), 3))   # 0.15, below one
guesses = [decode(z) for w in long_sent]        # ask the decoder once per position
print("distinct guesses for 20 positions:", len(set(guesses)))   # 1: every position is the same
print("decoded sentence:", " ".join(guesses))   # the whole output collapsed to one word

L = np.arange(1, 21)                            # lengths 1 to 20
share = 3 / L                                   # floats available per token
plt.figure(figsize=(6, 3))                      # wide, short figure
plt.plot(L, share, marker="o", markersize=3, color="tab:red")   # the shrinking share
plt.axhline(1.0, color="black", linestyle="--", label="one float per token")   # the break-even line
plt.axvline(3, color="grey", linestyle=":", label="length where it crosses")   # the crossing point
plt.xlabel("input length (tokens)")             # x-axis label
plt.ylabel("floats available per token")        # y-axis label
plt.title("Past three tokens the encoder holds less than one number each")   # the takeaway
plt.legend()                                    # name the guide lines
plt.show()                                      # render the figure

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>([...] * 7)[:20]</code> — a twenty-token sentence built by repetition.</li><li><code>np.mean(...)</code> — still only three numbers come out.</li><li><code>z.shape[0] / len(long_sent)</code> — the per-token budget, 0.15 here.</li><li><code>[decode(z) for w in ...]</code> — the decoder is asked per position but sees only <code>z</code>.</li><li><code>len(set(guesses))</code> — the number of distinct outputs; 1 means total collapse.</li><li><code>plt.axhline(1.0, ...)</code> — the break-even line, one float per token.</li><li><code>plt.axvline(3, ...)</code> — the length where the budget drops below one.</li></ul>

</details>

## Your turn

Set the section 4 slider to 4 and to 6 and compare the three bars before you look at the title. Then change `vectors` so that `cat` and `sat` share a direction and explain why the number of distinct decodes can only go down.